In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'bronze'
SOURCE_TABLE_NAME = 'channel_group'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'silver'
TARGET_TABLE_NAME = 'channel_group'

In [0]:
df = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .select(
        F.upper(F.trim(F.col('trade_chnl_desc'))).alias('trade_channel'),
        F.upper(F.trim(F.col('trade_group_desc'))).alias('trade_group'),
        F.upper(F.trim(F.col('trade_type_desc'))).alias('trade_type')
    )
    .filter(F.col('trade_channel').isNotNull())
)

## Cardinality check

`dropDuplicates` on a natural key is only safe when that key functionally
determines the remaining attributes. A future file where one `trade_channel` carries two different
attribute sets fails here instead of silently keeping an arbitrary row.

In [0]:
df_invalid_mapping = (
    df
    .groupBy('trade_channel')
    .agg(F.countDistinct(F.struct('trade_group', 'trade_type')).alias('attribute_count'))
    .filter(F.col('attribute_count') > 1)
)

assert df_invalid_mapping.count() == 0, 'trade_channel does not uniquely determine its attributes'

In [0]:
df = df.dropDuplicates(['trade_channel'])

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')